<a href="https://colab.research.google.com/github/AMJAMAITHILI/DL_LAB/blob/main/WEEK12_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Character-Level RNN (Next Character)

In [10]:
import torch
import torch.nn as nn

# Sample data
text = "hello world"
chars = list(set(text))
char2idx = {ch:i for i,ch in enumerate(chars)}
idx2char = {i:ch for ch,i in char2idx.items()}

# Convert to indices
data = [char2idx[ch] for ch in text]

# Create sequences
seq_len = 4
inputs, targets = [], []
for i in range(len(data) - seq_len):
    inputs.append(data[i:i+seq_len])
    targets.append(data[i+1:i+seq_len+1])

X = torch.tensor(inputs)
y = torch.tensor(targets)

# Model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

model = CharRNN(len(chars), 32)

# One-hot encoding
def one_hot(x, vocab_size):
    return torch.eye(vocab_size)[x]

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(200):
    x_onehot = one_hot(X, len(chars)).float()
    output = model(x_onehot)

    loss = criterion(output.view(-1, len(chars)), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Loss:", loss.item())

Loss: 2.0482749938964844
Loss: 0.07041042298078537
Loss: 0.05494273453950882
Loss: 0.052545033395290375


In [11]:
def generate(model, start="hell", length=10):
    model.eval()
    input_seq = [char2idx[ch] for ch in start]

    for _ in range(length):
        x = one_hot(torch.tensor([input_seq[-4:]]), len(chars)).float()
        out = model(x)
        pred = torch.argmax(out[0, -1]).item()
        input_seq.append(pred)

    return ''.join([idx2char[i] for i in input_seq])

print(generate(model))

hello worldrrw


Word-Level Prediction (Next Word)

In [12]:
import torch
import torch.nn as nn
from collections import Counter

# Sample corpus
sentences = [
    "i love deep learning",
    "i love machine learning",
    "deep learning is powerful"
]

# Build vocabulary
words = " ".join(sentences).split()


vocab = list(set(words + ["<start>", "<end>"]))

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

In [13]:
seq_len = 3
inputs, targets = [], []

for sentence in sentences:
    tokens = sentence.split()
    encoded = [word2idx[w] for w in tokens]

    for i in range(len(encoded) - seq_len):
        inputs.append(encoded[i:i+seq_len])
        targets.append(encoded[i+seq_len])

X = torch.tensor(inputs)
y = torch.tensor(targets)

In [14]:
class WordRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # last timestep
        return out

model = WordRNN(len(vocab), 16, 32)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Loss:", loss.item())

Loss: 2.512932062149048
Loss: 0.0013741222210228443
Loss: 0.0008250087848864496
Loss: 0.0005947102326899767


In [16]:
def predict_next(model, text):
    model.eval()
    tokens = text.split()
    encoded = [word2idx[w] for w in tokens]

    x = torch.tensor([encoded])
    output = model(x)

    pred = torch.argmax(output).item()
    return idx2word[pred]

print(predict_next(model, "i love deep"))

learning


Sentence-Level Prediction (Sequence Generation)

In [17]:
sentences = [
    "<start> i love deep learning <end>",
    "<start> deep learning is powerful <end>"
]

In [18]:
class SentenceLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

model = SentenceLSTM(len(vocab), 32, 64)

In [19]:
for epoch in range(200):
    total_loss = 0

    for sentence in sentences:
        tokens = sentence.split()
        encoded = [word2idx[w] for w in tokens]

        inp = torch.tensor([encoded[:-1]])
        target = torch.tensor([encoded[1:]])

        output, _ = model(inp)

        loss = criterion(output.view(-1, len(vocab)), target.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print("Loss:", total_loss)

Loss: 4.353040933609009
Loss: 4.353040933609009
Loss: 4.353040933609009
Loss: 4.353040933609009


In [20]:
def generate_sentence(model, start_word="<start>", max_len=10):
    model.eval()
    words = [start_word]
    hidden = None

    for _ in range(max_len):
        x = torch.tensor([[word2idx[words[-1]]]])
        output, hidden = model(x, hidden)

        pred = torch.argmax(output[0, -1]).item()
        next_word = idx2word[pred]

        if next_word == "<end>":
            break

        words.append(next_word)

    return " ".join(words)

print(generate_sentence(model))

<start> <start> <start> <start> learning <start> learning <start> learning <start> learning


**LSTM**

In [21]:
import torch
import torch.nn as nn

sentences = [
    "i love deep learning",
    "deep learning is powerful",
    "i love machine learning"
]

# Build vocab
words = " ".join(sentences).split()
vocab = list(set(words))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

# Create sequences
seq_len = 3
inputs, targets = [], []

for s in sentences:
    tokens = [word2idx[w] for w in s.split()]
    for i in range(len(tokens) - seq_len):
        inputs.append(tokens[i:i+seq_len])
        targets.append(tokens[i+seq_len])

X = torch.tensor(inputs)
y = torch.tensor(targets)

In [22]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, (h, c) = self.lstm(x)
        out = self.fc(out[:, -1, :])  # last timestep
        return out

model = LSTMModel(len(vocab), 32, 64)

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("LSTM Loss:", loss.item())

LSTM Loss: 1.9699708223342896
LSTM Loss: 1.9947312466683798e-05
LSTM Loss: 1.6371161109418608e-05
LSTM Loss: 1.4940692381060217e-05


In [24]:
def predict_lstm(text):
    tokens = [word2idx[w] for w in text.split()]
    x = torch.tensor([tokens])
    out = model(x)
    pred = torch.argmax(out).item()
    return idx2word[pred]

print(predict_lstm("i love deep"))

learning


**GRU Model**

In [25]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, h = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out

gru_model = GRUModel(len(vocab), 32, 64)

In [26]:
optimizer = torch.optim.Adam(gru_model.parameters(), lr=0.01)

for epoch in range(200):
    output = gru_model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("GRU Loss:", loss.item())

GRU Loss: 1.8854244947433472
GRU Loss: 2.7695816243067384e-05
GRU Loss: 2.3324944777414203e-05
GRU Loss: 2.1377913071773946e-05


In [27]:
def predict_gru(text):
    tokens = [word2idx[w] for w in text.split()]
    x = torch.tensor([tokens])
    out = gru_model(x)
    pred = torch.argmax(out).item()
    return idx2word[pred]

print(predict_gru("deep learning is"))

powerful


**Encoder–Decoder**

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1. DATA
# =========================

sentences = [
    ("i love you", "je t aime"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste")
]

# Build vocab
def build_vocab(sentences):
    words = set()
    for s in sentences:
        words.update(s.split())

    word2idx = {w:i+2 for i,w in enumerate(words)}
    word2idx["<pad>"] = 0
    word2idx["<start>"] = 1
    word2idx["<end>"] = len(word2idx)

    return word2idx

src_vocab = build_vocab([s[0] for s in sentences])
tgt_vocab = build_vocab([s[1] for s in sentences])

idx2word = {i:w for w,i in tgt_vocab.items()}

# Encode sentence
def encode(sentence, vocab):
    return [vocab["<start>"]] + [vocab[w] for w in sentence.split()] + [vocab["<end>"]]

data = []
for src, tgt in sentences:
    data.append((
        torch.tensor(encode(src, src_vocab)),
        torch.tensor(encode(tgt, tgt_vocab))
    ))

# =========================
# 2. ENCODER
# =========================

class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(input_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell


# =========================
# 3. DECODER
# =========================

class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell):
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x, (hidden, cell))
        output = self.fc(output)
        return output, hidden, cell


# =========================
# 4. SEQ2SEQ
# =========================

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        _, hidden, cell = self.encoder(src)

        outputs = []
        x = tgt[:, 0].unsqueeze(1)  # <start>

        for t in range(1, tgt.shape[1]):
            output, hidden, cell = self.decoder(x, hidden, cell)
            outputs.append(output)

            x = tgt[:, t].unsqueeze(1)  # teacher forcing

        return torch.cat(outputs, dim=1)


# =========================
# 5. MODEL INIT
# =========================

encoder = Encoder(len(src_vocab), 32, 64)
decoder = Decoder(len(tgt_vocab), 32, 64)
model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# =========================
# 6. TRAINING
# =========================

for epoch in range(300):
    total_loss = 0

    for src, tgt in data:
        src = src.unsqueeze(0)
        tgt = tgt.unsqueeze(0)

        output = model(src, tgt)

        loss = criterion(
            output.view(-1, len(tgt_vocab)),
            tgt[:, 1:].reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print("Loss:", total_loss)


# =========================
# 7. PREDICTION
# =========================

def translate(sentence):
    model.eval()

    src = torch.tensor([encode(sentence, src_vocab)])
    _, hidden, cell = encoder(src)

    x = torch.tensor([[tgt_vocab["<start>"]]])
    result = []

    for _ in range(10):
        output, hidden, cell = decoder(x, hidden, cell)

        pred = torch.argmax(output, dim=2).item()
        word = idx2word[pred]

        if word == "<end>":
            break

        result.append(word)
        x = torch.tensor([[pred]])

    return " ".join(result)


# =========================
# 8. TEST
# =========================

print("\nTranslation:")
print(translate("i love you"))

Loss: 6.2946072816848755
Loss: 0.006785796722397208
Loss: 0.0023646948975510895
Loss: 0.0012666041438933462
Loss: 0.0008047199080465361
Loss: 0.000561582637601532

Translation:
je t aime


**ATTENTION MECHANISM IMPLEMENTATION**

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1. DATA PREPARATION
# =========================

# Toy dataset (English → French)
pairs = [
    ("i love you", "je t aime"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste")
]

# Build vocabulary
def build_vocab(sentences):
    words = set()
    for s in sentences:
        words.update(s.split())

    word2idx = {w:i+2 for i,w in enumerate(words)}
    word2idx["<pad>"] = 0
    word2idx["<start>"] = 1
    word2idx["<end>"] = len(word2idx)

    return word2idx

src_vocab = build_vocab([p[0] for p in pairs])
tgt_vocab = build_vocab([p[1] for p in pairs])

idx2tgt = {i:w for w,i in tgt_vocab.items()}

# Convert sentence → tensor
def encode(sentence, vocab):
    tokens = sentence.split()
    return [vocab["<start>"]] + [vocab[w] for w in tokens] + [vocab["<end>"]]

data = []
for src, tgt in pairs:
    data.append((
        torch.tensor(encode(src, src_vocab)),
        torch.tensor(encode(tgt, tgt_vocab))
    ))

# =========================
# 2. ENCODER
# =========================

class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(input_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)

        # outputs → all hidden states (used for attention)
        outputs, (hidden, cell) = self.lstm(x)

        return outputs, hidden, cell


# =========================
# 3. ATTENTION MECHANISM
# =========================

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        # Linear layers to compute attention scores
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        # encoder_outputs: (batch, seq_len, hidden)
        seq_len = encoder_outputs.shape[1]

        # Repeat decoder hidden state for each time step
        hidden = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)

        # Concatenate hidden + encoder outputs
        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )

        # Compute attention scores
        scores = self.v(energy).squeeze(2)

        # Convert to probabilities
        attn_weights = torch.softmax(scores, dim=1)

        return attn_weights


# =========================
# 4. DECODER WITH ATTENTION
# =========================

class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(output_size, embed_size)
        self.attention = Attention(hidden_size)

        # LSTM input = embedding + context vector
        self.lstm = nn.LSTM(hidden_size + embed_size, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        # x shape: (batch, 1)
        x = self.embedding(x)

        # Get attention weights
        attn_weights = self.attention(hidden, encoder_outputs)

        # Compute context vector
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)

        # Concatenate embedding + context
        x = torch.cat((x, context), dim=2)

        # Pass through LSTM
        output, (hidden, cell) = self.lstm(x, (hidden, cell))

        # Final prediction
        output = self.fc(output)

        return output, hidden, cell


# =========================
# 5. SEQ2SEQ MODEL
# =========================

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        # Encode input sequence
        encoder_outputs, hidden, cell = self.encoder(src)

        outputs = []

        # First input to decoder = <start>
        x = tgt[:, 0].unsqueeze(1)

        # Loop over target sequence
        for t in range(1, tgt.shape[1]):
            output, hidden, cell = self.decoder(
                x, hidden, cell, encoder_outputs
            )

            outputs.append(output)

            # Teacher forcing
            x = tgt[:, t].unsqueeze(1)

        return torch.cat(outputs, dim=1)


# =========================
# 6. MODEL INITIALIZATION
# =========================

embed_size = 32
hidden_size = 64

encoder = Encoder(len(src_vocab), embed_size, hidden_size)
decoder = Decoder(len(tgt_vocab), embed_size, hidden_size)

model = Seq2Seq(encoder, decoder)

# =========================
# 7. TRAINING
# =========================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(300):
    total_loss = 0

    for src, tgt in data:
        src = src.unsqueeze(0)
        tgt = tgt.unsqueeze(0)

        output = model(src, tgt)

        loss = criterion(
            output.view(-1, len(tgt_vocab)),
            tgt[:, 1:].reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print("Loss:", total_loss)


# =========================
# 8. INFERENCE (TRANSLATION)
# =========================

def translate(sentence):
    model.eval()

    src = torch.tensor([encode(sentence, src_vocab)])
    encoder_outputs, hidden, cell = encoder(src)

    x = torch.tensor([[tgt_vocab["<start>"]]])
    result = []

    for _ in range(10):
        output, hidden, cell = decoder(x, hidden, cell, encoder_outputs)

        pred = output.argmax(2).item()
        word = idx2tgt[pred]

        if word == "<end>":
            break

        result.append(word)
        x = torch.tensor([[pred]])

    return " ".join(result)


# Test
print(translate("i love you"))

Loss: 6.44266939163208
Loss: 0.00525909336283803
Loss: 0.0020534578943625093
Loss: 0.0011004791886080056
Loss: 0.0007099037029547617
Loss: 0.0005024884012527764
je t aime


**BERT**

BERT for Next Word Prediction

In [30]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# Input sentence (predict next word)
text = "I love deep [MASK]"
inputs = tokenizer(text, return_tensors="pt")

# Get predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Find index of [MASK]
mask_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# Get predicted word
predicted_token_id = logits[0, mask_index].argmax(dim=-1)
predicted_word = tokenizer.decode(predicted_token_id)

print("Predicted word:", predicted_word)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicted word: .


Missing Word Prediction (Fill Mask)

In [31]:
from transformers import pipeline

# Fill-mask pipeline
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Input sentence
result = fill_mask("I love [MASK] learning")

# Show top predictions
for r in result:
    print(r['token_str'], ":", r['score'])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


his : 0.09320288151502609
just : 0.09277985990047455
her : 0.08704253286123276
about : 0.0784253254532814
the : 0.07103611528873444


Review Classification (Sentiment Analysis)

USING MODEL

In [33]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

# Load tokenizer + model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Input text
text = "I love this product!"

# Tokenize
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Predict
outputs = model(**inputs)
logits = outputs.logits

# Convert to probabilities
probs = F.softmax(logits, dim=-1)

# Get class
pred = torch.argmax(probs).item()

print("Sentiment:", "Positive" if pred == 1 else "Negative")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sentiment: Negative


USING PIPELINE

In [34]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

text = "I hate this product!"
result = classifier(text)[0]

print("Prediction:", result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Prediction: {'label': 'NEGATIVE', 'score': 0.9997503161430359}


**Implement GAN model on MNIST**

In [35]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Reshape, Flatten
from tensorflow.keras.optimizers import Adam

# Load and normalize MNIST data
(X_train, _), _ = mnist.load_data()
X_train = (X_train - 127.5) / 127.5  # Scale to [-1, 1]
X_train = X_train.reshape(-1, 784)

# Optimizer
opt = Adam(0.0002, 0.5)

# --- Generator function ---
def build_generator():
    model = Sequential([
        Dense(128, input_dim=100),
        LeakyReLU(0.2),
        Dense(784, activation='tanh'),  # Output shape: 28x28 = 784
    ])
    return model

# --- Discriminator function ---
def build_discriminator():
    model = Sequential([
        Dense(128, input_shape=(784,)),
        LeakyReLU(0.2),
        Dense(1, activation='sigmoid')  # Binary output: real (1) or fake (0)
    ])
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build models
generator = build_generator()
discriminator = build_discriminator()

# --- Train GAN ---
noise = np.random.normal(0, 1, (128, 100))          # Random noise
fake_images = generator.predict(noise)              # Generate fake images

real_images = X_train[np.random.randint(0, X_train.shape[0], 128)]  # Sample real images

# Labels for training
real_labels = np.ones((128, 1))
fake_labels = np.zeros((128, 1))

# Train discriminator
d_loss_real = discriminator.train_on_batch(real_images, real_labels)
d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

# Print discriminator performance
print("Discriminator accuracy on real:", d_loss_real[1]*100)
print("Discriminator accuracy on fake:", d_loss_fake[1]*100)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 
Discriminator accuracy on real: 0.0
Discriminator accuracy on fake: 26.953125


**Implement ViT model**

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# -----------------------------
# Patch Embedding
# -----------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


# -----------------------------
# Transformer Encoder Block
# -----------------------------
class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim)
        )

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_output, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_output

        x_norm = self.norm2(x)
        x = x + self.mlp(x_norm)

        return x


# -----------------------------
# Vision Transformer
# -----------------------------
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        embed_dim=256,
        depth=6,
        num_heads=8
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size, patch_size, in_channels, embed_dim
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))

        self.layers = nn.ModuleList([
            TransformerEncoder(embed_dim, num_heads)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed

        x = x.transpose(0, 1)

        for layer in self.layers:
            x = layer(x)

        x = x.transpose(0, 1)

        x = self.norm(x)

        cls_output = x[:, 0]

        return self.head(cls_output)


# -----------------------------
# Dataset (CIFAR-10)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)


# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# Model
# -----------------------------
model = VisionTransformer().to(device)


# -----------------------------
# Training Setup
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


# -----------------------------
# Training Loop
# -----------------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


# -----------------------------
# Evaluation
# -----------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 1.8261
